In [38]:
import numpy as np
import pandas as pd
import arff
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from autofeat import AutoFeatRegressor 
import featuretools as ft
from feature_engine.creation import MathFeatures

ModuleNotFoundError: No module named 'feature_engine'

In [3]:
df = pd.read_csv('diabetes.csv')


In [4]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [5]:
X = df.drop(columns=["Outcome"])  
y = df["Outcome"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Précision : {accuracy_score(y_test, y_pred):.2f}")
print(classification_report(y_test, y_pred))
print("Matrice de confusion :\n", confusion_matrix(y_test, y_pred))

Précision : 0.75
              precision    recall  f1-score   support

           0       0.81      0.80      0.81        99
           1       0.65      0.67      0.66        55

    accuracy                           0.75       154
   macro avg       0.73      0.74      0.73       154
weighted avg       0.76      0.75      0.75       154

Matrice de confusion :
 [[79 20]
 [18 37]]


In [13]:
#https://medium.com/@boukamchahamdi/autofeat-automating-feature-engineering-with-python-f22ec23265a9
af = AutoFeatRegressor( feateng_steps=2,n_jobs=-1)  

X_train_af = af.fit_transform(X_train, y_train)
X_test_af = af.transform(X_test)
X_train_af.head()
print(f"Nombre de nouvelles features créées : {X_train_af.shape[1] - X_train.shape[1]}")

model_af = LogisticRegression()
model_af.fit(X_train_af, y_train)
y_pred_af = model_af.predict(X_test_af)

print("\n Performance APRÈS AutoFeat")
print(f"Précision : {accuracy_score(y_test, y_pred_af):.2f}")
print(classification_report(y_test, y_pred_af))
print("Matrice de confusion :\n", confusion_matrix(y_test, y_pred_af))

c:\Users\H P\git\projetTutoreAutoFE\venv\lib\site-packages\autofeat\featsel.py:270: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  if np.max(np.abs(correlations[c].ravel()[:i])) < 0.9:


Nombre de nouvelles features créées : 8

 Performance APRÈS AutoFeat
Précision : 0.75
              precision    recall  f1-score   support

           0       0.79      0.82      0.81        99
           1       0.65      0.62      0.64        55

    accuracy                           0.75       154
   macro avg       0.72      0.72      0.72       154
weighted avg       0.74      0.75      0.75       154

Matrice de confusion :
 [[81 18]
 [21 34]]


c:\Users\H P\git\projetTutoreAutoFE\venv\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [33]:
es = ft.EntitySet(id="diabetes")
es = es.add_dataframe(dataframe_name="patients", dataframe=pd.DataFrame(X, columns=X.columns), index="index")

# Définir les features

feature_defs, feature_matrix = ft.dfs(
    entityset=es,
    target_dataframe_name="patients",
    verbose=True
)

print(f"Nombre de nouvelles features créées avec Featuretools : {len(feature_matrix) - X.shape[1]}")

# Convertir en DataFrame
X_ft = feature_defs.copy()

X_train_ft, X_test_ft, y_train, y_test  = train_test_split(X_ft, y, test_size=0.2, random_state=42)

print("\n Performance APRÈS featurestools")
print(f"Précision : {accuracy_score(y_test, y_pred_af):.2f}")
print(classification_report(y_test, y_pred_af))
print("Matrice de confusion :\n", confusion_matrix(y_test, y_pred_af))

model_ft = LogisticRegression()
model_ft.fit(X_train_ft, y_train)
y_pred_ft = model_ft.predict(X_test_ft)


print("\n📊 Performance APRÈS Featuretools")
print(f"Précision : {accuracy_score(y_test, y_pred_ft):.2f}")
print(classification_report(y_test, y_pred_ft))
print("Matrice de confusion :\n", confusion_matrix(y_test, y_pred_ft))


c:\Users\H P\git\projetTutoreAutoFE\venv\lib\site-packages\featuretools\entityset\entityset.py:1733: UserWarning: index index not found in dataframe, creating new integer column
  warnings.warn(
c:\Users\H P\git\projetTutoreAutoFE\venv\lib\site-packages\featuretools\synthesis\deep_feature_synthesis.py:169: UserWarning: Only one dataframe in entityset, changing max_depth to 1 since deeper features cannot be created
  warnings.warn(


Built 8 features
Elapsed: 00:00 | Progress:   0%|          

Elapsed: 00:00 | Progress: 100%|██████████
Nombre de nouvelles features créées avec Featuretools : 0

 Performance APRÈS featurestools
Précision : 0.75
              precision    recall  f1-score   support

           0       0.79      0.82      0.81        99
           1       0.65      0.62      0.64        55

    accuracy                           0.75       154
   macro avg       0.72      0.72      0.72       154
weighted avg       0.74      0.75      0.75       154

Matrice de confusion :
 [[81 18]
 [21 34]]

📊 Performance APRÈS Featuretools
Précision : 0.75
              precision    recall  f1-score   support

           0       0.81      0.79      0.80        99
           1       0.64      0.67      0.65        55

    accuracy                           0.75       154
   macro avg       0.73      0.73      0.73       154
weighted avg       0.75      0.75      0.75       154

Matrice de confusion :
 [[78 21]
 [18 37]]


c:\Users\H P\git\projetTutoreAutoFE\venv\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [36]:
!pip install feature-engine


In [ ]:
# Sélectionner les colonnes numériques
cols = ["Glucose", "BloodPressure", "BMI", "Age"]

# Appliquer des opérations mathématiques (addition, soustraction, multiplication...)
math_transformer = MathFeatures(
    variables=cols, operations=["sum", "prod", "mean", "std"]
)
df_new = math_transformer.fit_transform(df)